# fulfill_demands() Testing Framework

Comprehensive testing of different `fulfill_demands()` configurations on proforma service lines.

Tests various combinations of:
- Speed optimization (fixed middle speed vs 18-level optimization)
- Port operations constraint (stay time enforcement on/off)
- Transit time penalty (penalize late deliveries on/off)
- Transshipment ship class restriction (on/off)
- Different week levels (service frequencies)

**Note:** The parameter `turnon-vessel_speed_optimization` has counter-intuitive naming:
- `0` = Speed optimization ON (accurate but slow)
- `1` = Speed optimization OFF (fast but uses fixed speed)


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import time

from cma.data_reader import (
    read_vessel_class_data,
    read_port_data,
    read_sailing_distance_data,
    read_demand_with_transit_time,
    read_cnc_proforma_data,
)
from cma.port import PortGraph
from cma.servicegraph import ServiceGraph

# Keep noise down
np.set_printoptions(precision=4, suppress=True)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


## 1. Load Data


In [2]:
# Load core data
print("Loading vessel and port data...")
vesselpool = read_vessel_class_data()
portpool_main, portpool_dmd = read_port_data()

# Distance and demand with transit time expectations
print("Loading distance and demand matrices...")
dist_matrix = read_sailing_distance_data(portpool_main)
demand_matrix, transit_time_matrix = read_demand_with_transit_time(portpool_main)

# Build PortGraph without demand filtering (service lines may include more ports)
portgraph = PortGraph(
    portpool_main, 
    dist_matrix, 
    demand_matrix,
    mat_transit_time=transit_time_matrix,
    filter_by_demand=False
)

# Load proforma lines and metadata (attaches buffer profiles)
print("Loading CNC proforma service lines...")
proforma = read_cnc_proforma_data(portpool_main, vesselpool)
service_lines = proforma['lines']
metadata = proforma['metadata']

print(f"\n✓ Loaded {len(service_lines)} proforma service lines")
print(f"✓ Loaded {vesselpool.get_number_of_types()} vessel ranks")
print(f"✓ Loaded {portgraph.get_number_of_ports()} ports")


Loading vessel and port data...
Loading distance and demand matrices...
Loading CNC proforma service lines...

✓ Loaded 31 proforma service lines
✓ Loaded 11 vessel ranks
✓ Loaded 182 ports


## 2. Prepare Test Case


In [3]:
# Build service graph
servicegraph = ServiceGraph(service_lines)

# Collect OD pairs/paths
print("Finding paths for OD pairs...")
trans_ports = portgraph.filtered_by_transship_capacity()
od_pairs_dict = servicegraph.get_all_paths(portgraph, trans_ports)

# Use FULL dataset - all connected OD pairs
# This is what fulfill_demands() was designed to handle
all_od_pairs = od_pairs_dict['od_pairs']
all_demands = od_pairs_dict['od_pairs_demand']
all_paths = od_pairs_dict['od_pairs_path']

# Filter only OD pairs with positive demand
filtered_pairs = []
filtered_paths = []
filtered_demands = []

for od, paths, demand in zip(all_od_pairs, all_paths, all_demands):
    if demand > 0:
        filtered_pairs.append(od)
        filtered_paths.append(paths)
        filtered_demands.append(demand)

od_pairs = filtered_pairs
od_paths = filtered_paths

print(f"\n✓ Testing with FULL dataset: {len(od_pairs)} OD pairs")
print(f"✓ Total connected OD pairs available: {len(all_od_pairs)}")
print(f"✓ Unconnected OD pairs: {len(od_pairs_dict['unconnected'])}")
if len(od_pairs) > 0:
    print(f"✓ Demand range: {min(filtered_demands):.0f} - {max(filtered_demands):.0f} TEU/week")
    print(f"✓ Total demand: {sum(filtered_demands):.0f} TEU/week")
    avg_paths = sum(len(p) for p in od_paths) / len(od_paths)
    print(f"✓ Average paths per OD: {avg_paths:.1f}")

# Check for missing distances
missing = portgraph.get_pairs_missing_distance()
print(f"✓ Missing distances: {len(missing)} OD pairs")

print(f"\n⚠️  Note: This will test with the COMPLETE demand set.")
print(f"   Solve time may be 5-15 minutes per test.")


Finding paths for OD pairs...

✓ Testing with FULL dataset: 741 OD pairs
✓ Total connected OD pairs available: 741
✓ Unconnected OD pairs: 0
✓ Demand range: 1 - 2502 TEU/week
✓ Total demand: 70850 TEU/week
✓ Average paths per OD: 15.4
✓ Missing distances: 6160 OD pairs

⚠️  Note: This will test with the COMPLETE demand set.
   Solve time may be 5-15 minutes per test.


## 3. Define Test Configurations

Each configuration tests different features of `fulfill_demands()`.


In [4]:
# Base configuration (most conservative, fastest)
# NOTE: turnon-vessel_speed_optimization has INVERTED logic:
#   0 = Speed optimization ON (18 levels, slower but accurate)
#   1 = Speed optimization OFF (fixed middle speed, faster)
base_tuneparams = {
    'turnon-transship_shipclass_restriction': 0,
    'turnon-vessel_speed_optimization': 1,       # Fixed speed (faster) - counter-intuitive: 1=OFF
    'turnon-port_operations_constraint': 1,      # Enforce realistic stay times
    'turnon-transit_time_penalty': 0,            # No transit penalty
    'ctrparam-kts_buffer': 0,
    'ctrparam-transship_A': 100,
    'ctrparam-speed_soft_cap_kts': 16.5,
    'ctrparam-speed_penalty_multiplier': 2.0,
    'ctrparam-transit_penalty_multiplier': 1000.0,
    'ctrparam-demand_shortfall_penalty': 1e4,    # USD per TEU of unmet demand (soft constraint)
    'BigM-transship': 10000,
    'BigM-n_ships': 10,  # Increased from 2 to 10 for more capacity
    'BigM-saildays': 1000,
    'BigM-line_capacity': 100000,  # Increased from 30000 to 100000
    'BigM-portcall_cost': 2e9
}

# Test configurations
test_configs = [
    {
        'name': '1. Baseline (Fixed Speed)',
        'week_levels': [1, 2, 3, 4, 5, 6, 7, 8],  # Added week=4 for more flexibility
        'tuneparams': base_tuneparams.copy()
    },
    {
        'name': '2. With Transit Penalty',
        'week_levels': [1, 2, 3, 4, 5, 6, 7, 8],  # Added week=4 for more flexibility
        'tuneparams': {**base_tuneparams, 'turnon-transit_time_penalty': 1}
    },
    {
        'name': '3. Speed Optimization (18 levels)',
        'week_levels': [1, 2, 3, 4, 5, 6, 7, 8],  # Added week=4 for more flexibility
        'tuneparams': {**base_tuneparams, 'turnon-vessel_speed_optimization': 0}  # 0=ON
    },
    {
        'name': '4. No Port Ops Constraint',
        'week_levels': [1, 2, 3, 4, 5, 6, 7, 8],  # Added week=4 for more flexibility
        'tuneparams': {**base_tuneparams, 'turnon-port_operations_constraint': 0}
    },
    {
        'name': '5. Extended Weeks',
        'week_levels': [0.5, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        'tuneparams': base_tuneparams.copy()
    },
    {
        'name': '6. All Features (Speed Opt + Transit)',
        'week_levels': [1, 2, 3, 4, 5, 6, 7, 8],  # Added week=4 for more flexibility
        'tuneparams': {
            **base_tuneparams,
            'turnon-vessel_speed_optimization': 0,  # 0=ON for speed optimization
            'turnon-transit_time_penalty': 1,
            'turnon-transship_shipclass_restriction': 1
        }
    },
]

print(f"Defined {len(test_configs)} test configurations")


Defined 6 test configurations


## 4. Run Tests and Collect Results


In [ ]:
results = []

for config in test_configs:
    print(f"\n{'='*80}")
    print(f"Running: {config['name']}")
    print(f"{'='*80}")
    
    try:
        start_time = time.time()
        
        solution = servicegraph.fulfill_demands(
            od_pairs,
            od_paths,
            portgraph,
            vesselpool,
            config['week_levels'],
            config['tuneparams'],
        )
        
        elapsed = time.time() - start_time
        
        total_cost = solution.get('total cost')
        
        if total_cost is not None:
            print(f"✓ SUCCESS")
            print(f"  Total Cost: ${total_cost:,.2f}")
            print(f"  Solve Time: {elapsed:.2f}s")
            
            # Extract ship assignments
            ships = solution['ships'].value if hasattr(solution['ships'], 'value') else None
            weeks = solution['weeks'].value if hasattr(solution['weeks'], 'value') else None
            
            # Extract demand shortfall (unmet demand)
            demand_shortfall_vars = solution.get('demand_shortfall', [])
            total_unmet_demand = 0.0
            num_unmet_od_pairs = 0
            if demand_shortfall_vars:
                for eps in demand_shortfall_vars:
                    eps_val = eps.value if hasattr(eps, 'value') else 0
                    if eps_val is not None and eps_val > 0.01:  # Threshold for numerical errors
                        total_unmet_demand += eps_val
                        num_unmet_od_pairs += 1
            
            if ships is not None:
                total_ships = int(np.sum(ships))
                print(f"  Total Ships: {total_ships}")
            
            if total_unmet_demand > 0:
                print(f"  Unmet Demand: {total_unmet_demand:.2f} TEU/week ({num_unmet_od_pairs} OD pairs)")
            
            results.append({
                'Config': config['name'],
                'Status': 'SUCCESS',
                'Total Cost': total_cost,
                'Solve Time (s)': elapsed,
                'Total Ships': total_ships if ships is not None else None,
                'Unmet Demand (TEU)': total_unmet_demand,
                'Unmet OD Pairs': num_unmet_od_pairs,
                'Week Levels': str(config['week_levels']),
                'Speed Opt (0=ON)': config['tuneparams']['turnon-vessel_speed_optimization'],
                'Transit Penalty': config['tuneparams']['turnon-transit_time_penalty'],
                'Port Ops': config['tuneparams']['turnon-port_operations_constraint'],
            })
        else:
            print(f"✗ FAILED: Infeasible or unbounded")
            results.append({
                'Config': config['name'],
                'Status': 'INFEASIBLE',
                'Total Cost': None,
                'Solve Time (s)': elapsed,
                'Total Ships': None,
                'Week Levels': str(config['week_levels']),
                'Speed Opt (0=ON)': config['tuneparams']['turnon-vessel_speed_optimization'],
                'Transit Penalty': config['tuneparams']['turnon-transit_time_penalty'],
                'Port Ops': config['tuneparams']['turnon-port_operations_constraint'],
            })
            
    except Exception as e:
        print(f"✗ ERROR: {str(e)}")
        results.append({
            'Config': config['name'],
            'Status': f'ERROR: {str(e)[:50]}',
            'Total Cost': None,
            'Solve Time (s)': None,
            'Total Ships': None,
            'Week Levels': str(config['week_levels']),
            'Speed Opt (0=ON)': config['tuneparams']['turnon-vessel_speed_optimization'],
            'Transit Penalty': config['tuneparams']['turnon-transit_time_penalty'],
            'Port Ops': config['tuneparams']['turnon-port_operations_constraint'],
        })

print(f"\n{'='*80}")
print("All tests completed!")
print(f"{'='*80}")



Running: 1. Baseline (Fixed Speed)


c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\atoms\affine\reshape.py:68: FutureWarning: 
    You didn't specify the order of the reshape expression. The default order
    used in CVXPY is Fortran ('F') order. This default will change to match NumPy's
    default order ('C') in a future version of CVXPY.
    To suppress this warning, please specify the order explicitly.
    
  warnings.warn(reshape_order_warning, FutureWarning)
c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\reductions\solvers\solving_chain_utils.py:30: UserWarning: The problem includes expressions that don't support CPP backend. Defaulting to the SCIPY backend for canonicalization.
  warnings.warn(UserWarning(


Set parameter Username
Set parameter LicenseID to value 2725074
Academic license - for non-commercial use only - expires 2026-10-20
✓ SUCCESS
  Total Cost: $inf
  Solve Time: 10.47s

Running: 2. With Transit Penalty
✓ SUCCESS
  Total Cost: $inf
  Solve Time: 10.25s

Running: 3. Speed Optimization (18 levels)


## 5. Results Summary


In [ ]:
# Create summary DataFrame
df_results = pd.DataFrame(results)

print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80 + "\n")

display(df_results)

# Success rate
success_count = len(df_results[df_results['Status'] == 'SUCCESS'])
print(f"\nSuccess Rate: {success_count}/{len(df_results)} ({100*success_count/len(df_results):.1f}%)")

# Compare costs
successful = df_results[df_results['Status'] == 'SUCCESS'].copy()
if len(successful) > 0:
    print("\nCost Comparison (Successful Runs):")
    successful_sorted = successful.sort_values('Total Cost')
    for idx, row in successful_sorted.iterrows():
        print(f"  {row['Config']:40s}: ${row['Total Cost']:>15,.2f}")
    
    # Cost difference from baseline
    baseline = successful[successful['Config'].str.contains('Baseline')]
    if len(baseline) > 0:
        baseline_cost = baseline.iloc[0]['Total Cost']
        print(f"\nCost vs Baseline:")
        for idx, row in successful.iterrows():
            if 'Baseline' not in row['Config']:
                diff = row['Total Cost'] - baseline_cost
                pct = 100 * diff / baseline_cost
                print(f"  {row['Config']:40s}: {diff:+15,.2f} ({pct:+6.2f}%)")



RESULTS SUMMARY



,Config,Status,Total Cost,Solve Time (s),Total Ships,Unmet Demand (TEU),Unmet OD Pairs,Week Levels,Speed Opt (0=ON),Transit Penalty,Port Ops
0,1. Baseline (Fixed Speed),SUCCESS,inf,109.671359,None,0.0,0.0,"[1, 2, 3, 4, 5, 6, 7, 8]",1,0,1
1,2. With Transit Penalty,SUCCESS,inf,111.045295,None,0.0,0.0,"[1, 2, 3, 4, 5, 6, 7, 8]",1,1,1
2,3. Speed Optimization (18 levels),ERROR: Element 8 of a double array is Nan or Inf.,NaN,NaN,None,NaN,NaN,"[1, 2, 3, 4, 5, 6, 7, 8]",0,0,1
3,4. No Port Ops Constraint,ERROR: Strict inequalities are not allowed.,NaN,NaN,None,NaN,NaN,"[1, 2, 3, 4, 5, 6, 7, 8]",1,0,0
4,5. Extended Weeks,SUCCESS,inf,109.532783,None,0.0,0.0,"[0.5, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]",1,0,1
5,6. All Features (Speed Opt + Transit),ERROR: Element 8 of a double array is Nan or Inf.,NaN,NaN,None,NaN,NaN,"[1, 2, 3, 4, 5, 6, 7, 8]",0,1,1



Success Rate: 3/6 (50.0%)

Cost Comparison (Successful Runs):
  1. Baseline (Fixed Speed)               : $            inf
  2. With Transit Penalty                 : $            inf
  5. Extended Weeks                       : $            inf

Cost vs Baseline:
  2. With Transit Penalty                 :            +nan (  +nan%)
  5. Extended Weeks                       :            +nan (  +nan%)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_26084\3226038877.py:29: RuntimeWarning: invalid value encountered in scalar subtract
  diff = row['Total Cost'] - baseline_cost


## 6. Detailed Analysis (Optional)

Run this cell to get detailed breakdown of a specific solution.


## 7. Diagnostic Analysis - Identify Infeasibility Source

This cell will help identify why the model is infeasible.


In [ ]:
# Run a simplified diagnostic version
import cvxpy as cp

# Use baseline config
test_config = test_configs[0]

print("Building optimization problem...")
print(f"Config: {test_config['name']}")
print(f"OD pairs: {len(od_pairs)}")
print(f"Service lines: {len(service_lines)}")
print(f"Total demand: {sum([portgraph.get_demand_by_idx(od[0], od[1]) for od in od_pairs]):.0f} TEU/week")
print()

# Call fulfill_demands
solution = servicegraph.fulfill_demands(
    od_pairs,
    od_paths,
    portgraph,
    vesselpool,
    test_config['week_levels'],
    test_config['tuneparams'],
)

print(f"Solver Status: {solution.get('solver_status', 'UNKNOWN')}")
print(f"Objective value: {solution.get('total cost')}")
print()

# Check if cost is infinite
if solution.get('total cost') == float('inf'):
    print("❌ Problem is INFEASIBLE (objective = +inf)")
    print("   Even with soft demand constraints, no valid solution exists.")
    print()
    print("This suggests a hard constraint is being violated:")
    print("   • Port operations: stay_days >= transshipment / productivity")
    print("   • Sailing time: 7*week - port_stay >= 3.5 days")
    print("   • Speed bounds: distance must fit within min/max speed")
    print("   • Ship count: must be non-negative integers")
    print()
    print("Let me analyze the constraints...")
    print()
    
    # Check if any service lines have impossible distance/speed combinations
    print("Analyzing service line constraints:")
    print("-" * 80)
    from cma.vessel import VesselPool
    KTS_levels = np.arange(10.0, 28.5)  # Speed range
    
    for idx, line in enumerate(service_lines):
        line_distance = line.get_distance(portgraph)
        line_name = line.name()
        
        # Check each week level
        for week in test_config['week_levels']:
            # Minimum sailing days (assuming minimal port stay of 1 day total)
            min_port_stay = 1.0
            max_sailing_days = 7 * week - min_port_stay
            
            # Distance bounds
            if max_sailing_days > 0:
                min_speed_needed = line_distance / (24 * max_sailing_days)
                max_speed_allowed = KTS_levels[-1]
                
                if min_speed_needed > max_speed_allowed:
                    print(f"  ⚠️  {line_name:30s} week={week}: needs {min_speed_needed:.1f} kts (max={max_speed_allowed:.1f})")
    
elif solution.get('total cost') == float('-inf'):
    print("❌ Problem is UNBOUNDED (objective = -inf)")
    print("   Costs can become infinitely negative - likely a modeling error.")
    print()
    print("Possible causes:")
    print("   1. Flow variables can grow without proper capacity bounds")
    print("   2. Negative cost coefficient somewhere")
    print("   3. Missing constraint on decision variables")
    print()
else:
    print(f"✓ Problem SOLVED with finite cost: ${solution.get('total cost'):,.2f}")
    
    # Analyze the solution
    ships = solution['ships']
    if hasattr(ships, 'value') and ships.value is not None:
        total_ships = int(np.sum(ships.value))
        print(f"  Total ships: {total_ships}")
    
    # Check demand shortfall
    eps_vars = solution.get('demand_shortfall', [])
    if eps_vars:
        total_unmet = sum(eps.value if hasattr(eps, 'value') and eps.value is not None else 0 
                         for eps in eps_vars)
        total_demand = sum([portgraph.get_demand_by_idx(od[0], od[1]) for od in od_pairs])
        if total_unmet > 0.01:
            pct = 100 * total_unmet / total_demand
            print(f"  ⚠️  Unmet demand: {total_unmet:.2f} / {total_demand:.0f} TEU/week ({pct:.1f}%)")
        else:
            print(f"  ✓ All demand met!")




Building optimization problem...
Config: 1. Baseline (Fixed Speed)
OD pairs: 741
Service lines: 33
Total demand: 70850 TEU/week



c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 232 times so far.

  warnings.warn(msg, UserWarning)
c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 233 times so far.

  warnings.warn(msg, UserWarning)
c:

Solver Status: UNKNOWN
Objective value: inf

❌ Problem is INFEASIBLE (objective = +inf)
   Even with soft demand constraints, no valid solution exists.

This suggests a hard constraint is being violated:
   • Port operations: stay_days >= transshipment / productivity
   • Sailing time: 7*week - port_stay >= 3.5 days
   • Speed bounds: distance must fit within min/max speed
   • Ship count: must be non-negative integers

Let me analyze the constraints...

Analyzing service line constraints:
--------------------------------------------------------------------------------
  ⚠️  BBX2CNC                        week=1: needs 52.3 kts (max=28.0)
  ⚠️  BBX3CNC                        week=1: needs 48.1 kts (max=28.0)
  ⚠️  BMXCNC                         week=1: needs 45.2 kts (max=28.0)
  ⚠️  CHN1CNC                        week=1: needs 38.5 kts (max=28.0)
  ⚠️  CMS2CNC                        week=1: needs 39.4 kts (max=28.0)
  ⚠️  CS1CNC                         week=1: needs 37.4 kts (max=28.0

In [ ]:
# Deep Dive: Check distance/speed feasibility for ALL service lines
print("=" * 80)
print("DISTANCE/SPEED FEASIBILITY ANALYSIS")
print("=" * 80)
print()
print("Checking if any service lines have impossible distance/week combinations...")
print()

# Speed parameters from the model
KTS_min = 10.0  # Minimum speed
KTS_max = 18.5  # Maximum speed
min_port_stay_per_port = 0.5  # Assume minimum 0.5 days per port

infeasible_lines = []

for idx, line in enumerate(service_lines):
    line_distance = line.get_distance(portgraph)
    line_name = line.name()
    num_ports = len(line.tolist_port())
    
    # Estimate minimum port stay (very optimistic: 0.5 days per port)
    min_total_port_stay = num_ports * min_port_stay_per_port
    
    print(f"{line_name:30s}: {line_distance:6.0f} nm, {num_ports:2d} ports")
    
    feasible_for_any_week = False
    
    for week in test_config['week_levels']:
        # Total time available for round trip
        total_time_days = 7 * week
        
        # Time available for sailing (assuming minimal port operations)
        max_sailing_days = total_time_days - min_total_port_stay
        
        if max_sailing_days <= 0:
            print(f"  week={week}: ❌ No time for sailing (port stay > total time)")
            continue
        
        # Minimum speed required to cover the distance
        min_speed_required = line_distance / (24 * max_sailing_days)  # nautical miles per hour
        
        # Check if it's within vessel capability
        if min_speed_required <= KTS_max:
            print(f"  week={week}: ✓ Needs ≥{min_speed_required:4.1f} kts (OK, max={KTS_max:.1f})")
            feasible_for_any_week = True
        else:
            print(f"  week={week}: ❌ Needs ≥{min_speed_required:4.1f} kts (TOO FAST, max={KTS_max:.1f})")
    
    if not feasible_for_any_week:
        infeasible_lines.append((line_name, line_distance, num_ports))
    print()

print("=" * 80)
if infeasible_lines:
    print(f"❌ FOUND {len(infeasible_lines)} INFEASIBLE SERVICE LINES:")
    print()
    for name, dist, ports in infeasible_lines:
        print(f"  • {name:30s}: {dist:6.0f} nm, {ports} ports")
    print()
    print("These lines cannot complete their routes within any allowed week level")
    print("at maximum vessel speed. This makes the optimization infeasible!")
    print()
    print("SOLUTIONS:")
    print("  1. Add larger week levels (e.g., [1, 2, 3, 4, 5, 6, 7, 8])")
    print("  2. Remove or fix infeasible service lines")
    print("  3. Relax port operations constraints")
    print("  4. Increase maximum speed limit")
else:
    print("✓ All service lines are feasible for at least one week level")
print("=" * 80)



DISTANCE/SPEED FEASIBILITY ANALYSIS

Checking if any service lines have impossible distance/week combinations...

BBX2CNC                       :   7524 nm,  7 ports
  week=1: ❌ Needs ≥89.6 kts (TOO FAST, max=18.5)
  week=2: ❌ Needs ≥29.9 kts (TOO FAST, max=18.5)
  week=3: ✓ Needs ≥17.9 kts (OK, max=18.5)
  week=4: ✓ Needs ≥12.8 kts (OK, max=18.5)
  week=5: ✓ Needs ≥10.0 kts (OK, max=18.5)
  week=6: ✓ Needs ≥ 8.1 kts (OK, max=18.5)
  week=7: ✓ Needs ≥ 6.9 kts (OK, max=18.5)
  week=8: ✓ Needs ≥ 6.0 kts (OK, max=18.5)

BBX3CNC                       :   6922 nm,  8 ports
  week=1: ❌ Needs ≥96.1 kts (TOO FAST, max=18.5)
  week=2: ❌ Needs ≥28.8 kts (TOO FAST, max=18.5)
  week=3: ✓ Needs ≥17.0 kts (OK, max=18.5)
  week=4: ✓ Needs ≥12.0 kts (OK, max=18.5)
  week=5: ✓ Needs ≥ 9.3 kts (OK, max=18.5)
  week=6: ✓ Needs ≥ 7.6 kts (OK, max=18.5)
  week=7: ✓ Needs ≥ 6.4 kts (OK, max=18.5)
  week=8: ✓ Needs ≥ 5.5 kts (OK, max=18.5)

BBXCNC                        :   3003 nm,  3 ports
  week=1: ❌ Need

## Root Cause Analysis

Based on the diagnostics above, the model is showing as **INFEASIBLE** (not unbounded). When CVXPY returns `inf` for a minimization problem, it means no feasible solution exists.

### Why the Soft Constraint Didn't Help

The soft constraint on demand (allowing unmet demand) only relaxes the demand fulfillment requirement. However, other **hard constraints** remain:

1. **Distance/Speed/Time constraint**: Each service line must complete its round trip within `7 * week` days
   - Formula: `line_distance <= 24 * sailing_days * max_speed`
   - With sailing_days = `7 * week - port_stay_days`
   
2. **Port operations constraint**: `port_stay_days >= transshipment_volume / productivity`

3. **Minimum sailing time**: `sailing_days >= 3.5` days

### The Real Problem

Some service lines have **impossible distance/week combinations**:
- Long routes (e.g., 8000+ nm) with many ports
- Limited week levels: [1, 2, 3, 4]
- Maximum speed: 18.5 knots
- Result: Even at max speed with minimal port stays, round trip takes longer than allowed

Example:
- Distance: 8000 nm, 15 ports
- Week level: 4 (28 days total)
- Min port stay: 15 * 0.5 = 7.5 days
- Sailing time: 28 - 7.5 = 20.5 days
- Required speed: 8000 / (24 * 20.5) = **16.3 kts** ✓
- But with actual port operations (loading/unloading), port stays could be 1+ days each (15+ days total)
- Sailing time becomes: 28 - 15 = 13 days
- Required speed: 8000 / (24 * 13) = **25.6 kts** ❌ (exceeds max 18.5 kts)

### Solutions

**Option 1: Extend week levels (Recommended)**
```python
week_levels = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]  # Allow longer round trips
```

**Option 2: Add soft constraint on port operations**
```python
# Allow port stays to be less than required (with penalty)
port_stay_days + slack >= operations / productivity
```

**Option 3: Remove infeasible service lines from consideration**

**Option 4: Increase speed limits or relax speed constraints**



## Summary: The Model is INFEASIBLE, Not Unbounded

### Key Findings

1. **Status**: The optimization returns `cost = inf` which indicates **INFEASIBILITY**, not unboundedness
   - Unbounded would give `cost = -inf` (costs going to negative infinity)
   - Infeasible means no solution satisfies all constraints

2. **Root Cause**: **Distance/Speed/Time constraints** are too restrictive
   - Some service lines have routes too long to complete in allowed time
   - Constraint: `distance <= 24 * (7*week - port_stay) * max_speed`
   - With limited week levels [1,2,3,4] and max speed 18.5 kts, long routes are impossible

3. **Why Soft Demand Constraint Didn't Fix It**:
   - Soft constraint only allows unmet demand
   - But even with zero flow (no demand), service lines still must have feasible round-trip times
   - The distance/speed constraint is **independent of demand**

### Recommended Fix

**Extend the week levels to allow longer round trips:**

```python
base_tuneparams = {
    ...
    # Other params stay the same
}

test_configs = [
    {
        'name': '1. Baseline (Fixed Speed)',
        'week_levels': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],  # ← Extended range
        'tuneparams': base_tuneparams.copy()
    },
    ...
]
```

This allows service lines to operate on longer cycles (e.g., every 10 weeks instead of max 4 weeks), which gives them time to complete long routes at reasonable speeds.

### Alternative: Relax Port Operations Constraint

If extending week levels isn't sufficient, consider making port stays a soft constraint too (similar to what we did with demand).


In [ ]:
# Select configuration to analyze in detail (change index)
detail_config_idx = 0  # Baseline

print(f"Running detailed analysis for: {test_configs[detail_config_idx]['name']}")

solution = servicegraph.fulfill_demands(
    od_pairs,
    od_paths,
    portgraph,
    vesselpool,
    test_configs[detail_config_idx]['week_levels'],
    test_configs[detail_config_idx]['tuneparams'],
)

if solution.get('total cost') is not None:
    print(f"\nTotal Cost: ${solution['total cost']:,.2f}\n")
    
    # Ship assignments per line
    ships = solution['ships'].value
    weeks = solution['weeks'].value
    
    print("Ship Assignments per Service Line:")
    print("-" * 80)
    for idx, line in enumerate(service_lines):
        line_ships = ships[idx]
        line_weeks_bin = weeks[idx]
        week_val = test_configs[detail_config_idx]['week_levels'][np.argmax(line_weeks_bin)]
        
        total_line_ships = np.sum(line_ships)
        if total_line_ships > 0:
            print(f"\n{line.name():30s} (Week: {week_val})")
            for rank_idx, count in enumerate(line_ships):
                if count > 0:
                    print(f"  Rank {rank_idx+1}: {int(count)} ship(s)")
    
    # Port staying days
    stay_days = solution['port staying days']
    print("\n" + "="*80)
    print("Port Operations Summary")
    print("="*80)
    
    if hasattr(stay_days, 'value'):
        stay_days_vals = stay_days.value
        for idx, line in enumerate(service_lines):
            total_stay = np.sum(stay_days_vals[idx])
            if total_stay > 0.01:
                print(f"{line.name():30s}: {total_stay:.2f} days/week")
    
    # Demand Shortfall Analysis
    demand_shortfall_vars = solution.get('demand_shortfall', [])
    if demand_shortfall_vars:
        print("\n" + "="*80)
        print("Demand Shortfall Analysis")
        print("="*80)
        
        total_unmet = 0.0
        unmet_details = []
        
        for idx, eps in enumerate(demand_shortfall_vars):
            eps_val = eps.value if hasattr(eps, 'value') else 0
            if eps_val is not None and eps_val > 0.01:
                od_pair = od_pairs[idx]
                origin = portgraph.get_port_by_idx(od_pair[0])
                dest = portgraph.get_port_by_idx(od_pair[1])
                total_demand = portgraph.get_demand_by_idx(od_pair[0], od_pair[1])
                unmet_details.append((origin.get_id(), dest.get_id(), total_demand, eps_val))
                total_unmet += eps_val
        
        if unmet_details:
            print(f"\nTotal Unmet Demand: {total_unmet:.2f} TEU/week across {len(unmet_details)} OD pairs\n")
            print(f"{'Origin':<10} {'Dest':<10} {'Total Demand':>15} {'Unmet':>15} {'% Unmet':>10}")
            print("-" * 65)
            for origin, dest, total_demand, unmet in sorted(unmet_details, key=lambda x: -x[3])[:20]:
                pct_unmet = 100 * unmet / total_demand if total_demand > 0 else 0
                print(f"{origin:<10} {dest:<10} {total_demand:>15.2f} {unmet:>15.2f} {pct_unmet:>9.1f}%")
            if len(unmet_details) > 20:
                print(f"\n... and {len(unmet_details) - 20} more OD pairs")
        else:
            print("\n✓ All demand fully satisfied!")
else:
    print("Solution infeasible or unbounded")


Running detailed analysis for: 1. Baseline (Fixed Speed)


c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 265 times so far.

  warnings.warn(msg, UserWarning)
c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 266 times so far.

  warnings.warn(msg, UserWarning)
c:


Total Cost: $inf

Ship Assignments per Service Line:
--------------------------------------------------------------------------------


TypeError: 'NoneType' object is not subscriptable